In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from pathlib import Path
from collections import Counter

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# pytorch modules and dependencies
import torch
import torchvision
from torchvision.transforms import v2
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

# load, explore and analyze the data provided

## load and analyze the train.csv

In [ ]:
ROOT = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")

# load the train csv data and describe
train = pd.read_csv(ROOT / "train.csv")
print(train.shape)
display(train.head(5))

# get the labels
LABELS = [c for c in train.columns if c not in ("StudyInstanceUID", "Report")]
print("\n", len(LABELS), LABELS)

In [ ]:
# Distinguish fully labelled, partially labelled, and unlabelled studies.
label_values = train[LABELS]
fully_labelled = label_values.notna().all(axis=1)
any_label = label_values.notna().any(axis=1)
partially_labelled = any_label & ~fully_labelled
unlabelled = ~any_label

print("Fully labelled studies:", int(fully_labelled.sum()))
print("Partially labelled studies:", int(partially_labelled.sum()))
print("Studies with no structured labels:", int(unlabelled.sum()))
print("Missing values by label:")
display(
    label_values.isna().sum().sort_values(ascending=False).to_frame("missing_count")
)

In [ ]:
import matplotlib.pyplot as plt

# Label prevalence and imbalance: compute only where the relevant label is known.
known_counts = label_values.notna().sum()
positive_counts = (label_values == 1).sum()
negative_counts = (label_values == 0).sum()
label_audit = pd.DataFrame({
    "known": known_counts,
    "positive": positive_counts,
    "negative": negative_counts,
    "positive_fraction_among_known": positive_counts / known_counts.replace(0, np.nan),
})
display(label_audit.sort_values("positive_fraction_among_known"))
label_audit["positive_fraction_among_known"].sort_values().plot.barh(
    figsize=(8, 6), title="Positive fraction among known labels"
)
plt.tight_layout()

In [ ]:
# explore reports for potential weak-label extraction
print("Missing reports:", int(train["Report"].isna().sum()))
for i in range(min(5, len(train))):
    print(f"\nReport sample {i} | study={train.loc[i, 'StudyInstanceUID']}")
    print(str(train.loc[i, "Report"])[:1000])

## load and analyze the train_series.csv

In [ ]:
# load and describe the train_series data
train_series = pd.read_csv(ROOT / "train_series.csv")
print(train_series.shape)
display(train_series.head(5))
print("\nStudies in train_series table: ", train_series["StudyInstanceUID"].nunique())
print("\nSeries per study summary:")
display(
    train_series.groupby("StudyInstanceUID").size().describe().to_frame("series_per_study")
)

# anatomical plane
print("\nSeries counts by plane:")
display(
    train_series["Anatomical_Plane"].value_counts(dropna=False).to_frame("series_count")
)

In [ ]:
# compare fluid sensitive and fat suppression columns to check for similarity
print(pd.crosstab(train_series["Fluid_Sensitive"], train_series["Fat_Suppression"]))

# check how many studies contain at least on fluid_sensitive series in each plane
fs = train_series[train_series["Fluid_Sensitive"] == 1]
print("\nStudies with fluid_sensitive series by plane:")
print(fs.groupby("Anatomical_Plane")["StudyInstanceUID"].nunique())

### load, explore and analyze the DICOM slices in the train_series folder

In [ ]:
import pydicom

ROOT = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")

def series_dir(split_dir, study_uid, series_uid):
    return ROOT/split_dir/study_uid/series_uid

sample = train_series.iloc[0]
sample_dir = series_dir("train_series", sample.StudyInstanceUID, sample.SeriesInstanceUID)
sample_files = list(sample_dir.glob("*.dcm"))
print("Example study:", sample.StudyInstanceUID)
print("Example series:", sample.SeriesInstanceUID)
print("DICOM files found:", len(sample_files))
if sample_files:
    ds = pydicom.dcmread(sample_files[0], stop_before_pixels=True)
    for tag in ["Modality", "SeriesDescription", "Rows", "Columns", "SliceThickness",
                "PixelSpacing", "RepetitionTime", "EchoTime", "ImageType"]:
        print(f"{tag:20s}", getattr(ds, tag, "n/a"))

In [ ]:
# load, display and compile a complete series into a single NumPy array
def load_series(dir_path):
    files = list(Path(dir_path).glob("*.dcm"))
    if not files:
        raise FileNotFoundError(f"No DICOM slices found in {dir_path}")
        
    dss = [
        pydicom.dcmread(f)
        for f in Path(dir_path).glob("*.dcm")
    ]

    # sort the slices in anatomical order
    try:
        iop = np.asarray(dss[0].ImageOrientationPatient, dtype=np.float64)
        normal = np.cross(iop[:3], iop[3:])

        dss.sort(
            key=lambda x: float(
                np.dot(
                    normal,
                    np.asarray(x.ImagePositionPatient, dtype=np.float64)
                )
            )
        )
    except (AttributeError, ValueError, TypeError):
        dss.sort(
            key=lambda ds: int(
                getattr(x, "InstanceNumber", 0)
            )
        )

    # Apply DICOM rescale where present; handle MONOCHROME1 so intensities have consistent polarity.
    slices = []
    for ds in dss:
        image = ds.pixel_array.astype(np.float32)
        image = image * float(getattr(ds, "RescaleSlope", 1.0)) + float(getattr(ds, "RescaleIntercept", 0.0))
        if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
            image = image.max() - image
        slices.append(image)
    return np.stack(slices), dss

vol, sample_datasets = load_series(sample_dir)
print("Volume shape (no. of slices, image height, image width: ", vol.shape)

In [ ]:
import matplotlib.pyplot as plt

# displaying a few select slices for visual inspection
n = min(8, len(vol))
idx = np.linspace(0, len(vol) - 1, n).astype(int)
fig, axes = plt.subplots(1, n, figsize=(2.2 * n, 2.5))
for ax, i in zip(np.atleast_1d(axes), idx):
    ax.imshow(vol[1], cmap="grey");
    ax.set_title(f"slice {i}");
plt.suptitle(f"{sample.Anatomical_Plane} | fluid={sample.Fluid_Sensitive} fat_sup={sample.Fat_Suppression}")
plt.show()

## Build a complete series manifest

In [ ]:
def count_dicom_slices(split_dir, study_uid, series_uid):
    return len(list(series_dir(split_dir, study_uid, series_uid).glob("*.dcm")))

series_manifest = train_series.copy()
series_manifest["n_slices"] = [
    count_dicom_slices("train_series", row.StudyInstanceUID, row.SeriesInstanceUID)
    for row in series_manifest.itertuples(index=False)
]
series_manifest["series_path_exists"] = [
    series_dir("train_series", row.StudyInstanceUID, row.SeriesInstanceUID).is_dir()
    for row in series_manifest.itertuples(index=False)
]

study_manifest = train[["StudyInstanceUID"]].copy()
study_manifest["fully_labelled"] = fully_labelled.to_numpy()
study_manifest["partially_labelled"] = partially_labelled.to_numpy()
study_manifest["unlabelled"] = unlabelled.to_numpy()
study_manifest = study_manifest.merge(
    series_manifest.groupby("StudyInstanceUID").agg(
        n_series=("SeriesInstanceUID", "nunique"),
        total_slices=("n_slices", "sum"),
        missing_series_dirs=("series_path_exists", lambda x: int((~x).sum())),
    ),
    on="StudyInstanceUID", how="left"
)
print("Study inventory:", study_manifest.shape)
display(study_manifest.head())
print("Series directories missing:", int((~series_manifest["series_path_exists"]).sum()))
print("Series with zero slices:", int((series_manifest["n_slices"] == 0).sum()))
print("Slice-count distribution:")
display(series_manifest["n_slices"].describe().to_frame("slices_per_series"))
print("Series counts by plane and fluid-sensitive flag:")
display(pd.crosstab(series_manifest["Anatomical_Plane"], series_manifest["Fluid_Sensitive"], dropna=False))

# Save manifests for reproducibility
series_manifest.to_csv("/kaggle/working/series_manifest.csv", index=False)
study_manifest.to_csv("/kaggle/working/study_manifest.csv", index=False)

# prepare the data for pytorch

In [ ]:
from torch.utils.data import Dataset

def normalize_volume(vol):
    # Per-volume robust scaling
    lo, hi = np.percentile(vol, [0.5, 99.5])
    vol = np.clip(vol, lo, hi)
    return (vol - lo) / max(float(hi - lo), 1e-6)

class KneeStudyDataset(Dataset):
    def __init__(self, study_ids, series_df, split_dir="train_series",
                 labels_df=None):
        self.ids = list(study_ids)
        self.series_df = series_df
        self.split_dir = split_dir
        self.labels = None if labels_df is None else labels_df.set_index("StudyInstanceUID")
        self.series_by_study = {
            sid: g.copy() for sid, g in series_df.groupby("StudyInstanceUID")
        }

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        sid = self.ids[i]
        rows = self.series_by_study.get(sid)
        if rows is None or rows.empty:
            raise RuntimeError(f"No series metadata for study {sid}")

        series_volumes, series_info = [], []
        for row in rows.itertuples(index=False):
            path = series_dir(self.split_dir, sid, row.SeriesInstanceUID)
            if not path.is_dir() or not any(path.glob("*.dcm")):
                raise FileNotFoundError(f"Missing DICOM series directory/files: {path}")
            vol, _ = load_series(path)
            vol = normalize_volume(vol)
            # Keeping native H x W resolution
            vol = torch.from_numpy(vol.astype(np.float32))  # (slices, H, W)
            series_volumes.append(vol)
            series_info.append({
                "series_uid": row.SeriesInstanceUID,
                "plane": getattr(row, "Anatomical_Plane", None),
                "fluid_sensitive": getattr(row, "Fluid_Sensitive", None),
                "fat_suppression": getattr(row, "Fat_Suppression", None),
                "n_slices": int(vol.shape[0]),
            })

        item = {"study_uid": sid, "series": series_volumes, "series_info": series_info}
        if self.labels is not None:
            y = self.labels.loc[sid, LABELS].to_numpy(dtype=np.float32)
            item["y"] = torch.tensor(y, dtype=torch.float32)
            item["label_known"] = torch.tensor(~np.isnan(y), dtype=torch.bool)
        return item

def collate_one_study(batch):
    # Each study has a different number of series/slices, retaining each as a list.
    if len(batch) != 1:
        raise ValueError("Use batch_size=1 with collate_one_study for variable-length studies.")
    return batch[0]

## testing the complete data pipeline

In [ ]:
from torch.utils.data import DataLoader

# Partially labelled studies are retained: label_known marks which targets are usable.
supervised_ids = train.loc[train[LABELS].notna().any(axis=1), "StudyInstanceUID"].tolist()
unlabelled_ids = train.loc[train[LABELS].isna().all(axis=1), "StudyInstanceUID"].tolist()

# Using labels_df=train for partially labelled studies; missing targets are masked by label_known.
ds_train = KneeStudyDataset(
    supervised_ids, series_manifest, split_dir="train_series",
    labels_df=train
)
loader = DataLoader(
    ds_train, batch_size=1, shuffle=True, num_workers=0,
    collate_fn=collate_one_study
)

# Smoke-testing one complete study (all available series, all slices).
batch = next(iter(loader))
print("Study:", batch["study_uid"])
print("Number of series retained:", len(batch["series"]))
print("Slices per series:", [tuple(v.shape) for v in batch["series"]])
print("Labels shape:", tuple(batch["y"].shape))
print("Known labels:", int(batch["label_known"].sum()), "/", len(LABELS))

# Unlabelled studies remain available for future self-supervised/semi-supervised learning.
print("Unlabelled studies available for additional learning:", len(unlabelled_ids))

# Fine tune ResNet50

In [ ]:
# Training configuration
import random, copy, math, time
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
from torchvision import models
from torchvision.models import ResNet50_Weights

SEED = 117
IMAGE_SIZE = 320
SLICES_PER_SERIES_TRAIN = 4   # randomly re-sampled each epoch
SLICES_PER_SERIES_VALID = 8   # deterministic coverage for repeatable validation
EPOCHS = 8
LR_HEAD = 3e-4
LR_BACKBONE = 2e-5
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0              # DICOM dataset loads per study
CHECKPOINT_PATH = "/kaggle/working/resnet50_knee_best.pt"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# check if CUDA device is available if not use cpu
train_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training device:", train_device)
if train_device.type == "cuda":
    print(torch.cuda.get_device_name(0))

# Study-level split
all_supervised_ids = train.loc[train[LABELS].notna().any(axis=1), "StudyInstanceUID"].astype(str).tolist()
train_ids, valid_ids = train_test_split(
    all_supervised_ids, test_size=0.15, random_state=SEED, shuffle=True
)
print(f"Training studies: {len(train_ids):,} | validation studies: {len(valid_ids):,}")

In [ ]:
# Creating study items from all series. I reuse the existing DICOM reader and label mask.
train_ds = KneeStudyDataset(
    train_ids, series_manifest, split_dir="train_series", labels_df=train
)
valid_ds = KneeStudyDataset(
    valid_ids, series_manifest, split_dir="train_series", labels_df=train
)

# ImageNet normalization for pretrained ResNet-50. DICOM grayscale is repeated to 3 channels.
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32).view(1, 3, 1, 1)

def select_slice_indices(n, count, training):
    if n <= count:
        return np.arange(n, dtype=int)
    if training:
        return np.sort(np.random.choice(n, size=count, replace=False))
    # Deterministic, evenly distributed validation coverage across each series.
    return np.linspace(0, n - 1, count).round().astype(int)

def study_to_images(item, training):
    """Resize selected slices from every series; return (N,3,H,W), labels and known-mask."""
    chunks = []
    limit = SLICES_PER_SERIES_TRAIN if training else SLICES_PER_SERIES_VALID
    for volume in item["series"]:
        # volume shape: (slices, H, W), normalized robustly to [0, 1] by dataset
        indices = select_slice_indices(int(volume.shape[0]), limit, training)
        if len(indices) == 0:
            continue
        x = volume[indices].unsqueeze(1)  # (N,1,H,W)
        x = F.interpolate(x, size=(IMAGE_SIZE, IMAGE_SIZE), mode="bilinear", align_corners=False)
        x = x.repeat(1, 3, 1, 1)          # ResNet-50 expects 3 channels
        chunks.append(x)
    if not chunks:
        raise RuntimeError(f"Study {item['study_uid']} has no usable slices")
    x = torch.cat(chunks, dim=0)
    x = (x - IMAGENET_MEAN) / IMAGENET_STD
    return x, item["y"], item["label_known"]

# Quick shape check before the expensive training loop.
preview = train_ds[0]
preview_x, preview_y, preview_known = study_to_images(preview, training=False)
print("Example study:", preview["study_uid"])
print("Sampled images from all series:", tuple(preview_x.shape))
print("Label vector:", tuple(preview_y.shape), "| known labels:", int(preview_known.sum()))
del preview, preview_x

In [ ]:
# ResNet-50 outputs one logit per finding for each slice.
weights = ResNet50_Weights.DEFAULT
model = models.resnet50(weights=weights)
model.fc = torch.nn.Linear(model.fc.in_features, len(LABELS))
model = model.to(train_device)

# Freeze early layers for a stable first fine-tuning pass; layer4 and classifier remain trainable.
for p in model.parameters():
    p.requires_grad = False
for p in model.layer4.parameters():
    p.requires_grad = True
for p in model.fc.parameters():
    p.requires_grad = True

optimizer = torch.optim.AdamW([
    {"params": model.layer4.parameters(), "lr": LR_BACKBONE},
    {"params": model.fc.parameters(), "lr": LR_HEAD},
], weight_decay=WEIGHT_DECAY)

# Positive weighting uses only known labels in the training split.
train_label_rows = train.set_index("StudyInstanceUID").loc[train_ids, LABELS]
known_counts = train_label_rows.notna().sum(axis=0).to_numpy(dtype=np.float32)
pos_counts = (train_label_rows == 1).sum(axis=0).to_numpy(dtype=np.float32)
neg_counts = (train_label_rows == 0).sum(axis=0).to_numpy(dtype=np.float32)
pos_weight = np.divide(neg_counts, np.maximum(pos_counts, 1.0))
pos_weight = np.clip(pos_weight, 1.0, 20.0)
pos_weight_t = torch.tensor(pos_weight, dtype=torch.float32, device=train_device)
print("Label positive weights:")
display(pd.DataFrame({"label": LABELS, "known": known_counts, "positive": pos_counts,
                      "negative": neg_counts, "pos_weight": pos_weight}))

In [ ]:
from torch.utils.checkpoint import checkpoint

def forward_slice_chunks(images, chunk_size=16, use_checkpoint=False):
    """Running all selected slices in small batches to reduce peak GPU memory."""
    logits = []
    for chunk in images.split(chunk_size, dim=0):
        if use_checkpoint and torch.is_grad_enabled():
            z = checkpoint(model, chunk, use_reentrant=False)
        else:
            z = model(chunk)
        logits.append(z)
    return torch.cat(logits, dim=0)

def aggregate_slice_logits(slice_logits):
    """Aggregate slice logits into study logits with a per-label top-k mean.

    Top-k pooling is a simple multiple-instance-learning baseline: a finding may only
    be visible on a subset of slices. k grows with the number of slices, up to 8.
    """
    n = slice_logits.shape[0]
    k = max(1, min(8, int(math.ceil(0.10 * n))))
    return torch.topk(slice_logits, k=k, dim=0).values.mean(dim=0)

def masked_bce(study_logits, y, known):
    y = y.to(device=study_logits.device, dtype=torch.float32)
    known = known.to(device=study_logits.device, dtype=torch.bool)
    # Replace NaNs before BCE; mask them out of the loss afterwards.
    y_safe = torch.nan_to_num(y, nan=0.0)
    loss_matrix = F.binary_cross_entropy_with_logits(
        study_logits, y_safe, reduction="none", pos_weight=pos_weight_t
    )
    known_f = known.float()
    return (loss_matrix * known_f).sum() / known_f.sum().clamp_min(1.0)

@torch.no_grad()
def compute_metrics(y_true, y_prob, known_mask):
    aucs, aps = [], []
    per_label = []
    for j, name in enumerate(LABELS):
        m = known_mask[:, j].astype(bool)
        yt, yp = y_true[m, j], y_prob[m, j]
        auc = np.nan
        ap = np.nan
        if len(yt) and len(np.unique(yt)) == 2:
            auc = roc_auc_score(yt, yp)
            ap = average_precision_score(yt, yp)
            aucs.append(auc); aps.append(ap)
        per_label.append({"label": name, "known_n": int(m.sum()), "positive_n": int((yt == 1).sum()),
                          "AUROC": auc, "average_precision": ap})
    return (float(np.mean(aucs)) if aucs else float("nan"),
            float(np.mean(aps)) if aps else float("nan"), pd.DataFrame(per_label))

def run_epoch(dataset, training):
    model.train(training)
    losses = []
    ys, probs, masks = [], [], []
    order = np.random.permutation(len(dataset)) if training else np.arange(len(dataset))
    for step, idx in enumerate(order, 1):
        item = dataset[int(idx)]
        images, y, known = study_to_images(item, training=training)
        images = images.to(train_device, non_blocking=True)
        y = y.to(train_device)
        known = known.to(train_device)
        if training:
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(train_device.type == "cuda")):
                slice_logits = forward_slice_chunks(images, chunk_size=16, use_checkpoint=training)
                study_logits = aggregate_slice_logits(slice_logits)
                loss = masked_bce(study_logits, y, known)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
        else:
            with torch.no_grad():
                with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(train_device.type == "cuda")):
                    slice_logits = forward_slice_chunks(images, chunk_size=16, use_checkpoint=training)
                    study_logits = aggregate_slice_logits(slice_logits)
                    loss = masked_bce(study_logits, y, known)
        losses.append(float(loss.detach().cpu()))
        ys.append(torch.nan_to_num(y.detach().cpu(), nan=0.0).numpy())
        masks.append(known.detach().cpu().numpy().astype(bool))
        probs.append(torch.sigmoid(study_logits.detach().float()).cpu().numpy())
        if step % 25 == 0 or step == len(order):
            print(f"  {'train' if training else 'valid'} {step}/{len(order)} | running loss={np.mean(losses):.4f}", end="\r")
        del item, images, slice_logits, study_logits, loss
    print()
    y_all = np.stack(ys)
    p_all = np.stack(probs)
    m_all = np.stack(masks)
    auc, ap, table = compute_metrics(y_all, p_all, m_all)
    return float(np.mean(losses)), auc, ap, table

In [ ]:
best_val_loss = float("inf")
history = []
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    print(f"\nEpoch {epoch}/{EPOCHS}")
    train_loss, _, _, _ = run_epoch(train_ds, training=True)
    val_loss, val_auc, val_ap, val_table = run_epoch(valid_ds, training=False)
    elapsed = time.time() - t0
    row = {"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
           "val_macro_auroc": val_auc, "val_macro_average_precision": val_ap,
           "seconds": elapsed}
    history.append(row)
    print(row)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "labels": LABELS,
            "image_size": IMAGE_SIZE,
            "validation_loss": val_loss,
            "validation_macro_auroc": val_auc,
            "validation_macro_average_precision": val_ap,
        }, CHECKPOINT_PATH)
        print("Saved best checkpoint:", CHECKPOINT_PATH)

history_df = pd.DataFrame(history)
display(history_df)
val_table.to_csv("/kaggle/working/resnet50_validation_metrics_by_label.csv", index=False)
history_df.to_csv("/kaggle/working/resnet50_training_history.csv", index=False)
print("Best checkpoint:", CHECKPOINT_PATH)

## Run inference on the test data

In [ ]:
# Load test metadata. These files/directories should sit directly under ROOT.
test = pd.read_csv(ROOT / "test.csv")
test_series = pd.read_csv(ROOT / "test_series.csv")

required_test_cols = {"StudyInstanceUID"}
if not required_test_cols.issubset(test.columns):
    raise ValueError(f"test.csv must contain columns: {required_test_cols}; found {list(test.columns)}")
if not {"StudyInstanceUID", "SeriesInstanceUID"}.issubset(test_series.columns):
    raise ValueError("test_series.csv must contain StudyInstanceUID and SeriesInstanceUID columns")

test["StudyInstanceUID"] = test["StudyInstanceUID"].astype(str)
test_series["StudyInstanceUID"] = test_series["StudyInstanceUID"].astype(str)
test_series["SeriesInstanceUID"] = test_series["SeriesInstanceUID"].astype(str)
test_ids = test["StudyInstanceUID"].tolist()

missing_series_meta = sorted(set(test_ids) - set(test_series["StudyInstanceUID"]))
print("Test studies:", len(test_ids))
print("Test series metadata rows:", len(test_series))
print("Test studies without series metadata:", len(missing_series_meta))
if missing_series_meta:
    raise ValueError(f"Some test studies have no rows in test_series.csv, e.g. {missing_series_meta[:5]}")

# Audit DICOM directory presence before inference.
test_series_manifest = test_series.copy()
test_series_manifest["n_slices"] = [
    count_dicom_slices("test_series", row.StudyInstanceUID, row.SeriesInstanceUID)
    for row in test_series_manifest.itertuples(index=False)
]
test_series_manifest["series_path_exists"] = [
    series_dir("test_series", row.StudyInstanceUID, row.SeriesInstanceUID).is_dir()
    for row in test_series_manifest.itertuples(index=False)
]
display(test_series_manifest[["StudyInstanceUID", "SeriesInstanceUID", "n_slices", "series_path_exists"]].head())
print("Series with zero DICOM slices:", int((test_series_manifest["n_slices"] == 0).sum()))
if (test_series_manifest["n_slices"] == 0).any():
    raise FileNotFoundError("At least one test series has no readable *.dcm files; inspect test_series_manifest before inference.")


In [ ]:
# Use the same deterministic slice coverage as validation, without requiring labels.
def study_to_images_for_inference(item):
    chunks = []
    for volume in item["series"]:
        indices = select_slice_indices(int(volume.shape[0]), SLICES_PER_SERIES_VALID, training=False)
        if len(indices) == 0:
            continue
        x = volume[indices].unsqueeze(1)
        x = F.interpolate(x, size=(IMAGE_SIZE, IMAGE_SIZE), mode="bilinear", align_corners=False)
        x = x.repeat(1, 3, 1, 1)
        chunks.append(x)
    if not chunks:
        raise RuntimeError(f"Test study {item['study_uid']} has no usable slices")
    x = torch.cat(chunks, dim=0)
    mean = IMAGENET_MEAN.to(x.device)
    std = IMAGENET_STD.to(x.device)
    return (x - mean) / std

# The test dataset has no labels. It still loads every series and every slice from disk;
# inference samples evenly spaced slices from each series to fit model input/memory constraints.
test_ds = KneeStudyDataset(test_ids, test_series_manifest, split_dir="test_series", labels_df=None)

if not Path(CHECKPOINT_PATH).is_file():
    raise FileNotFoundError(f"Best model checkpoint not found: {CHECKPOINT_PATH}. Run the training cells first.")
checkpoint_data = torch.load(CHECKPOINT_PATH, map_location=train_device)
if checkpoint_data.get("labels") != LABELS:
    raise ValueError("Checkpoint label order differs from current LABELS; refusing to generate misaligned predictions.")
model.load_state_dict(checkpoint_data["model_state_dict"])
model.eval()

test_predictions = []
with torch.no_grad():
    for i in range(len(test_ds)):
        item = test_ds[i]
        images = study_to_images_for_inference(item).to(train_device, non_blocking=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(train_device.type == "cuda")):
            slice_logits = forward_slice_chunks(images, chunk_size=16, use_checkpoint=False)
            study_logits = aggregate_slice_logits(slice_logits)
        probs = torch.sigmoid(study_logits.float()).cpu().numpy()
        test_predictions.append(probs)
        if (i + 1) % 10 == 0 or (i + 1) == len(test_ds):
            print(f"Predicted {i + 1}/{len(test_ds)} test studies")

pred_matrix = np.vstack(test_predictions) if test_predictions else np.empty((0, len(LABELS)))
output = pd.DataFrame(pred_matrix, columns=LABELS)
output.insert(0, "StudyInstanceUID", test_ids)
output[LABELS] = output[LABELS].round(1)
output.to_csv("submission.csv", index=False)
print('Yor submission was successfully saved!')
output.head(10)